In [ ]:
import requests

# Bounding box más amplia: sur, oeste, norte, este
# Abarca Torrent, Alboraia, Picanya, etc.
query = """
[out:json][timeout:60];
(
  node["railway"="station"](39.3, -0.5, 39.6, -0.25);
  node["railway"="tram_stop"](39.3, -0.5, 39.6, -0.25);
);
out body;
"""

# Enviar consulta a Overpass
response = requests.post("http://overpass-api.de/api/interpreter", data={"data": query})
data = response.json()

# Procesar los resultados
stations = []
for element in data["elements"]:
    name = element.get("tags", {}).get("name", "Desconocido")
    lat = element["lat"]
    lon = element["lon"]
    stations.append((name, lat, lon))

# Mostrar los primeros resultados
for s in stations:
    print(f"{s[0]} - {s[1]} - {s[2]}")

print(f"\nTotal estaciones encontradas: {len(stations)}")

import csv

with open("estaciones_valencia.csv", mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Nombre", "Latitud", "Longitud"])
    for s in stations:
        writer.writerow(s)
    writer.writerow(("Àngel Guimerà", 39.4703606, -0.3836834))

Sant Isidre - 39.4510701 - -0.4026173
Vicent Zaragozà - 39.4833051 - -0.3572899
Universitat Politècnica - 39.4811835 - -0.3500137
la Carrasca - 39.479595 - -0.3444999
Tarongers Ernest Lluch - 39.4781633 - -0.3395337
Beteró - 39.476702 - -0.3344684
À Punt - 39.5121647 - -0.4246933
La Cadena - 39.4752445 - -0.3294173
Platja Malva-rosa - 39.473897 - -0.3256238
Platja les Arenes - 39.4690236 - -0.3256365
Dr. Lluch - 39.4684375 - -0.328127
Cabanyal - 39.472274 - -0.3275746
Benimaclet - 39.4849982 - -0.3630506
Trinitat - 39.4863151 - -0.3675372
Pont de Fusta - 39.4821884 - -0.372913
Empalme - 39.4994406 - -0.4022864
Palau de Congressos - 39.4972359 - -0.4000684
Garbí - 39.4921542 - -0.394133
Benicalap - 39.4899682 - -0.3900161
Trànsits - 39.4895533 - -0.3870385
Marxalenes - 39.4879552 - -0.3838592
Reus - 39.4859308 - -0.3817171
La Granja - 39.5039112 - -0.4123489
Parc Científic - 39.5152106 - -0.4224569
Francesc Cubells - 39.4633076 - -0.3341847
Canyamelar - 39.4664282 - -0.3279643
Masies - 

In [ ]:
import pandas as pd
import numpy as np

# Leer archivos
estaciones = pd.read_excel("estaciones_valencia.xlsx")
propiedades = pd.read_excel("propiedades_valencia_2025.xlsx")

# Arreglar decimales
estaciones["Latitud"] = estaciones["Latitud"].astype(str).str.replace(",", ".").astype(float)
estaciones["Longitud"] = estaciones["Longitud"].astype(str).str.replace(",", ".").astype(float)
propiedades["latitude"] = propiedades["latitude"].astype(str).str.replace(",", ".").astype(float)
propiedades["longitude"] = propiedades["longitude"].astype(str).str.replace(",", ".").astype(float)

# Función rápida para calcular distancia entre coordenadas (Haversine)
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371000  # radio Tierra en metros
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Para cada propiedad, calcular la distancia a todas las estaciones y tomar la mínima
def calcular_min_distancia(propiedades_df, estaciones_df):
    min_distancias = []
    for i, row in propiedades_df.iterrows():
        dists = haversine_np(
            row["latitude"],
            row["longitude"],
            estaciones_df["Latitud"].values,
            estaciones_df["Longitud"].values
        )
        min_distancias.append(dists.min())
    return min_distancias

# Calcular y añadir columna
propiedades["distancia_min_estacion_m"] = calcular_min_distancia(propiedades, estaciones)

# Guardar resultado
propiedades.to_excel("propiedades_valencia_2025.xlsx", index=False)
